# check

> Listen back with Su-śrotā: what a render says, how far it is from the text, the best of a few seeds, and word timings.

In [ ]:
#| default_exp check

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

A text-to-speech model sometimes mumbles, skips a word or garbles a line, and nobody can listen to thousands of renders. This module uses a speech recogniser to check them instead: it transcribes each render, measures how far the transcript is from the text, re-renders the bad ones, and finds where each word sits in the audio.

The recogniser is [Su-śrotā](https://huggingface.co/prathoshap/sushrota-sanskrit-asr), IndicConformer fine-tuned on śāstric Sanskrit. Its CTC output layer covers all 22 IndicConformer languages, and Sanskrit is the 256 columns starting at 4096, with the blank token at 5632. The checkpoint is configured for AI4Bharat's fork of NeMo (a `multilingual` tokenizer and a `multisoftmax` decoder), so `Ear` rewrites the configuration for the standard NeMo: the tokenizer type becomes `agg`, the settings only the fork understands are removed, and the weights are loaded without requiring every key to match. Only the encoder and the CTC output layer are used. Install it with `pip install 'vachaka[check]'`.

In [ ]:
#| export
import re, tarfile, unicodedata, numpy as np
from fastcore.utils import *
from vachaka.core import cache, clean, concat, Recitation

In [ ]:
#| export
ASR, ASR_FILE = 'prathoshap/sushrota-sanskrit-asr', 'sushrota_sanskrit_asr_v13b.nemo'
OFF, V, BL = 4096, 256, 5632

## Comparing text

Su-śrotā does not write exactly what the text says. It leaves out daṇḍas, digits and the avagraha (`ऽ`), and writes a word-final `म्` as an anusvāra (`ं`). `norm(s)` brings any text to that form, so a transcript can be compared with its source fairly. `cer(ref, hyp)` is the character error rate: the edit distance (`lev`, the number of single-character insertions, deletions and substitutions) between the two normalised texts, divided by the length of the reference. Spaces are ignored, because Sanskrit texts differ in where they split words joined by sandhi.

In [ ]:
#| export
_DROP = re.compile('[\u0966-\u096F।॥ऽॐ\u0951-\u0954\u1CD0-\u1CFF]')
_KEEP = re.compile('[^\u0900-\u097F\\s]')
def norm(s):
    "Text in the form Su-śrotā writes it: NFC, without accents, digits, daṇḍas or avagraha, and with word-final म् as anusvāra."
    s = _KEEP.sub(' ', _DROP.sub(' ', unicodedata.normalize('NFC', s)))
    s = re.sub('(.)(?=\\1)', lambda m: '' if unicodedata.category(m[1]) in ('Mn', 'Mc') else m[1], s)
    s = re.sub('म्(?=\\s|$)', 'ं', s)
    return re.sub('\\s+', ' ', s).strip()

In [ ]:
test_eq(norm('चोदनालक्षणोऽर्थो धर्मः ॥ २ ॥'), 'चोदनालक्षणो र्थो धर्मः')
test_eq(norm('कंसचाणूरमर्दनम्'), 'कंसचाणूरमर्दनं')

In [ ]:
#| export
def lev(a, b):
    "Edit distance between sequences `a` and `b`."
    row = list(range(len(b)+1))
    for i,x in enumerate(a, 1):
        prev, row[0] = row[0], i
        for j,y in enumerate(b, 1): prev, row[j] = row[j], min(row[j]+1, row[j-1]+1, prev+(x!=y))
    return row[-1]

def cer(ref, hyp):
    "The character error rate of `hyp` against `ref`, both normalised and with spaces ignored."
    r, h = [norm(o).replace(' ', '') for o in (ref, hyp)]
    return lev(r, h)/max(1, len(r))

In [ ]:
test_eq(lev('kitten', 'sitting'), 3)
test_eq(cer('अथातो धर्मजिज्ञासा', 'अथातो धर्मजिज्ञासा'), 0)
assert 0 < cer('तस्य निमित्तपरीष्टिः', 'तस्य निमित्तपरेष्टिः') < 0.1

## Ear

`Ear()` loads Su-śrotā on the CPU. `hear(audio, sr)` transcribes audio and returns the normalised text. `post(audio, sr)` returns the model's raw output for each 40 ms frame, the log-probability of the blank and of each of the 256 Sanskrit tokens, together with the frame length in milliseconds; `words` below uses it to align.

In [ ]:
#| export
def _cfg(nemo):
    "Su-śrotā's config rewritten for upstream NeMo, cached beside the vachaka sources."
    from omegaconf import OmegaConf, DictConfig
    p = cache/'sushrota_v13b.yaml'
    if p.exists(): return p
    with tarfile.open(nemo) as t: c = OmegaConf.create(t.extractfile('./model_config.yaml').read().decode())
    def strip(d):
        if not isinstance(d, DictConfig): return
        for k in list(d.keys()):
            if k in ('multisoftmax', 'multilingual', 'language_keys', 'return_language_id'): del d[k]
            else: strip(d[k])
    strip(c); c.tokenizer.type = 'agg'
    p.parent.mkdir(parents=True, exist_ok=True); OmegaConf.save(c, p)
    return p

In [ ]:
#| export
class Ear:
    "Su-śrotā on the CPU, for transcripts and word timings."
    def __init__(self):
        import logging, nemo.collections.asr as na
        from huggingface_hub import hf_hub_download
        logging.getLogger('nemo_logger').setLevel(logging.ERROR)
        p = hf_hub_download(ASR, ASR_FILE)
        self.m = na.models.EncDecHybridRNNTCTCBPEModel.restore_from(p, map_location='cpu', override_config_path=str(_cfg(p)), strict=False).eval()
        self.tok = self.m.tokenizer.tokenizers_dict['sa']

    def post(self, audio, sr):
        "Log-probabilities of the blank and the 256 Sanskrit tokens for each frame, and the milliseconds per frame."
        import torch, librosa
        w = librosa.resample(np.asarray(audio, np.float32), orig_sr=sr, target_sr=16000) if sr != 16000 else np.asarray(audio, np.float32)
        with torch.no_grad():
            enc, _ = self.m.forward(input_signal=torch.tensor(w)[None], input_signal_length=torch.tensor([len(w)]))
            lp = self.m.ctc_decoder(encoder_output=enc)[0].numpy()
        P = lp[:, [BL]+list(range(OFF, OFF+V))]
        P = P - np.logaddexp.reduce(P, axis=1, keepdims=True)
        return P, len(w)/16/len(P)

    def hear(self, audio, sr):
        "Transcribe `audio`, returning the normalised text Su-śrotā hears."
        ids, out, prev = self.post(audio, sr)[0].argmax(1), [], -1
        for i in ids:
            if i != prev and i: out.append(self.tok.ids_to_tokens([int(i)-1])[0])
            prev = i
        return norm(''.join(out).replace('▁', ' '))

## Best of a few seeds

Each render uses a random seed, and a different seed gives a slightly different performance. `best_of(render, ear, text)` calls `render(seed=...)` up to `tries` (3) times with different seeds, measures each result's character error rate against `text`, and keeps the best, stopping early once one is at or below `max_cer` (0.08). The rate is stored on the result as `Recitation.cer`.

`ear.heard(rec)` transcribes a render one piece at a time, using its `spans`, and joins the results. Transcribing the whole render at once fails on long runs of nearly identical lines, such as a tarpaṇa or the eight directions of a digvandana: the recogniser loses its place, and good renders scored between 0.47 and 0.76.

In [ ]:
#| export
@patch
def heard(self:Ear, rec):
    "Transcribe `rec` one piece of its `spans` at a time and join the transcripts."
    if not getattr(rec, 'spans', None): return self.hear(rec.audio, rec.sr)
    return ' '.join(self.hear(rec.audio[int(s*rec.sr/1000):int(e*rec.sr/1000)], rec.sr) for s,e in rec.spans)

In [ ]:
class _E(Ear):
    def __init__(self): self.got = []
    def hear(self, audio, sr): self.got.append(len(audio)); return 'क'
e, r = _E(), AttrDict(audio=np.zeros(16000*3, np.float32), sr=16000, spans=[(0, 1000), (1500, 3000)])
test_eq(e.heard(r), 'क क'); test_eq(e.got, [16000, 24000])
test_eq(_E().heard(AttrDict(audio=np.zeros(800, np.float32), sr=16000)), 'क')

In [ ]:
#| export
def best_of(render, ear, text, tries=3, max_cer=0.08, seed=60):
    "Call `render(seed=...)` up to `tries` times and keep the render whose transcript is closest to `text`."
    best = None
    for k in range(tries):
        r = render(seed=seed+7*k)
        r.cer = cer(text, ear.heard(r))
        if best is None or r.cer < best.cer: best = r
        if best.cer <= max_cer: break
    return best

## Word timings

`ear.words(rec, words)` finds the start and end of each word, given a render and, for each of its pieces, the list of words as displayed. It force-aligns them, matching the known word sequence against the audio with torchaudio's CTC aligner, inside each piece's own span, so a badly rendered line cannot drag the timings of the others. Each word is tokenised from its cleaned form. A word with no Sanskrit tokens, such as a lone daṇḍa, gets an empty span at the end of the word before it. The result has one list per piece, of `[word, start_ms, end_ms]`.

In [ ]:
#| export
@patch
def words(self:Ear, rec, words):
    "The `[word, start_ms, end_ms]` of each displayed word, for each piece of `rec`."
    import torch, torchaudio.functional as F
    P, fms = self.post(rec.audio, rec.sr)
    res = []
    for (s,e), ws in zip(rec.spans, words):
        a, b = int(s/fms), min(len(P), int(np.ceil(e/fms)))
        ids = [[i+1 for i in self.tok.text_to_ids(norm(clean(w)))] for w in ws]
        flat = [i for o in ids for i in o]
        if not flat or b-a < len(flat): res.append([[w, s, e] for w in ws]); continue
        ali, sc = F.forced_align(torch.tensor(np.ascontiguousarray(P[a:b]))[None], torch.tensor([flat]), blank=0)
        toks = F.merge_tokens(ali[0], sc[0].exp())
        out, k, end = [], 0, s
        for w,o in zip(ws, ids):
            if not o: out.append([w, end, end]); continue
            t0, t1 = toks[k], toks[k+len(o)-1]; k += len(o)
            out.append([w, s+round(t0.start*fms), s+round(t1.end*fms)]); end = out[-1][2]
        for i in range(len(out)-1): out[i][2] = max(out[i][2], out[i+1][1])
        res.append(out)
    return res

These cells need the Su-śrotā checkpoint (520 MB) and the Vāgdhenu weights, so they do not run in the tests.

In [ ]:
#| eval: false
from vachaka.core import Chanter
ch, ear = Chanter(), Ear()
lines = ['वसुदेवसुतं देवं कंसचाणूरमर्दनम्', 'देवकीपरमानन्दं कृष्णं वन्दे जगद्गुरुम्']
r = best_of(partial(ch.chant, lines), ear, ' '.join(lines))
r.cer, ear.words(r, [l.split() for l in lines])

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()